# Data preparation

Remove duplicate sentences from collections

In [47]:
import pandas as pd
from pathlib import Path

In [48]:
collection,genre_suffix = 'blb',''
if collection == 'blb':
  genre_suffix = '_with_genre'

maskedToken = 'machine'

dataPath = 'masking_data'
processedFolder = Path('input_data')
processedFolder.mkdir(exist_ok=True, parents=True)


In [49]:
# start from reprocessed data
data_files = list(Path(dataPath).glob(f'bl_microsoft_{maskedToken}*'))
data_dfs = [pd.read_json(f, lines=True) for f in data_files]
data_df = pd.concat(data_dfs, ignore_index=True).reset_index(drop=True)

  0%|          | 0/21 [00:58<?, ?it/s]
0it [00:25, ?it/s]


In [50]:
from transformers import pipeline
pipe = pipeline("fill-mask", model="Livingwithmachines/bert_1760_1900")
pipe("This [MASK] is awesome", top_k=20)

  0%|          | 0/68594 [01:24<?, ?it/s]


Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie bert.embeddings.word_embeddings.weight to cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie cls.predictions.bias to cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BertForMaskedLM LOAD REPORT from: Livingwithmachines/bert_1760_1900
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.6211618185043335,
  'token': 2173,
  'token_str': 'place',
  'sequence': 'this place is awesome'},
 {'score': 0.04975159838795662,
  'token': 2182,
  'token_str': 'here',
  'sequence': 'this here is awesome'},
 {'score': 0.01808903180062771,
  'token': 2112,
  'token_str': 'part',
  'sequence': 'this part is awesome'},
 {'score': 0.015941545367240906,
  'token': 2023,
  'token_str': 'this',
  'sequence': 'this this is awesome'},
 {'score': 0.014459202066063881,
  'token': 2035,
  'token_str': 'all',
  'sequence': 'this all is awesome'},
 {'score': 0.011959346942603588,
  'token': 2555,
  'token_str': 'region',
  'sequence': 'this region is awesome'},
 {'score': 0.011650588363409042,
  'token': 6019,
  'token_str': 'passage',
  'sequence': 'this passage is awesome'},
 {'score': 0.011316896416246891,
  'token': 2088,
  'token_str': 'world',
  'sequence': 'this world is awesome'},
 {'score': 0.010259662754833698,
  'token': 2029,
  'token_str': 'which',
  'sequence': 'this wh

In [52]:
data_df.columns

Index(['record_id', 'date', 'title', 'author', 'main_publication_country',
       'main_language', 'fiction_score', 'nonfiction_score', 'query',
       'prev_sentence', 'sentence', 'masked_sentence', 'next_sentence', 'pg',
       'ocr_quality_mean', 'ocr_quality_sd', 'mask_index', 'total_masks',
       'pred_bert_1760_1850', 'pred_bert_1890_1900', 'pred_bert_contemporary'],
      dtype='str')

In [54]:
from tqdm import tqdm
tqdm.pandas()
restults = data_df['masked_sentence'].apply(lambda x: pipe(x, top_k=20))

0it [00:00, ?it/s]
0it [00:21, ?it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (653 > 512). Running this sequence through the model will result in indexing errors


RuntimeError: The size of tensor a (653) must match the size of tensor b (512) at non-singleton dimension 1

In [ ]:

data_df.to_csv(f"./{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}.tsv", sep="\t")

In [29]:

data_df = pd.read_csv(f"./{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}.tsv", sep="\t", index_col=0)
data_df.shape

(79683, 21)

In [31]:
data_df.columns

Index(['record_id', 'date', 'title', 'author', 'main_publication_country',
       'main_language', 'fiction_score', 'nonfiction_score', 'query',
       'prev_sentence', 'sentence', 'masked_sentence', 'next_sentence', 'pg',
       'ocr_quality_mean', 'ocr_quality_sd', 'mask_index', 'total_masks',
       'pred_bert_1760_1850', 'pred_bert_1890_1900', 'pred_bert_contemporary'],
      dtype='str')

In [32]:
print(f'Before cleaning the sentences, we have {data_df.shape[0]} sentences for the target token {maskedToken}.')
data_df['sent_clean'] = data_df['sentence'].apply(lambda x: ''.join([ch.lower() for ch in x if ch.isalpha()]))
data_df.drop_duplicates(subset=['sent_clean'], inplace=True)
data_df.drop(columns=['sent_clean'], inplace=True)
data_df.reset_index(drop=True, inplace=True)
data_df.to_csv(f'{dataPath}/{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv', sep='\t')
print(f'Before cleaning the sentences, we have {data_df.shape[0]} sentences for the target token {maskedToken}.')

Before cleaning the sentences, we have 79683 sentences for the target token machine.
Before cleaning the sentences, we have 68594 sentences for the target token machine.


In [33]:
data_df.to_csv(f'{processedFolder}/{collection}_{maskedToken}_clusters{genre_suffix}_deduplicated.tsv', sep='\t')